# SFT results

What supervised finetuning does to a base model, run by run. Every number here
comes from the run's own `artifacts/logs/*.jsonl`, so a cell re-reads the
archive instead of re-training -- re-running this notebook costs seconds, not
GPU hours.

A new finetuning run appends a section. LoRA and DPO get their own notebooks.

The task interface is `task.py`; the loop is `sft.py`; the run is one line:

    python basic.py sft --task reverse --steps 1000 --batch-size 8 --grad-accum 2


## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import json
import math
import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.append(str(root / "src"))
sys.path.append(str(root / "src" / "video"))

import torch

from checkpoint import latest_ckpt, load_checkpoint
from generate import generate
from instruct import Instruct
from paths import CKPT_DIR, LOG_DIR
from reverse import Reverse
from tokenizer import ENDOFTEXT

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# label -> (log stem, training window). sft.py logs the window from 2026-09-21
# on; these three runs predate that line, so it is recorded by hand.
RUNS = {
    "scratch 11.2M": ("sft_smoke_2026-09-21_21-45-49", 128),
    "smoke base 30M": ("sft_reverse_2026-09-21_22-13-20", 256),
    "fineweb 123.6M": ("sft_reverse_2026-09-21_22-16-03", 256),
    "instruct 123.6M": ("sft_instruct_2026-09-21_23-21-46", 512),
    "instruct 27M ts": ("sft_instruct_2026-09-22_15-05-41", 512),
    "reverse 27M ts": ("sft_reverse_2026-09-22_15-46-43", 256),
}
CKPT = CKPT_DIR / "sft_reverse_2026-09-21_22-16-03.pt"
CKPT_INSTRUCT = CKPT_DIR / "sft_instruct_2026-09-21_23-21-46.pt"
CKPT_TS_INSTRUCT = CKPT_DIR / "sft_instruct_2026-09-22_15-05-41.pt"
CKPT_TS_REVERSE = CKPT_DIR / "sft_reverse_2026-09-22_15-46-43.pt"


def load_run(label):
    """(config, eval rows, summary) -- the jsonl a Run wrote is the archive."""
    stem, window = RUNS[label]
    lines = (LOG_DIR / f"{stem}.jsonl").read_text().splitlines()
    events = [json.loads(line) for line in lines]
    pick = lambda e: [x for x in events if x["event"] == e]
    return pick("config")[0] | {"window": window}, pick("log"), pick("summary")[0]


def table(rows, cols, w=12):
    print("".join(f"{c:>{w}}" for c in cols))
    for r in rows:
        print(
            "".join(
                f"{v:>{w}.4f}"
                if isinstance(v, float)
                else f"{'-' if v is None else v:>{w}}"
                for v in (r.get(c) for c in cols)
            )
        )

## 1. does SFT solve reverse, and how fast?

The 123.6M FineWeb-Edu base, finetuned on `Reverse` with flex + document masking.
`exact_match` is the scoreboard: a greedy decode of 200 fresh words, scored by
string equality. **Reads a log, < 1 s.**

Recorded result: **1.000 exact match by step 250**, 1000 steps in 4.6 min on a 4060.

| step | comp  | prompt | exact_match |
| ---- | ----- | ------ | ----------- |
| 0    | 6.067 | 8.728  | 0.000       |
| 50   | 0.432 | 8.019  | 0.400       |
| 100  | 0.007 | 8.797  | 0.995       |
| 250  | 0.001 | 9.059  | **1.000**   |
| 1000 | 0.000 | 9.816  | 1.000       |

Completion loss starts at **6.07, not ln(50259) = 10.8** — that gap is what
pretraining bought on a task the base has never seen. Everything past step 250 is
the prompt loss drifting; 300 steps is the honest setting.


In [3]:
cfg, rows, summ = load_run("fineweb 123.6M")
print(
    f"{cfg['n_embed']}d x {cfg['n_layer']}L   window {cfg['window']}   "
    f"batch {cfg['batch_size']}x{cfg['grad_accum_steps']}   lr {cfg['lr']}   "
    f"{cfg['attention']}"
)
table(rows, ["step", "comp", "prompt", "exact_match", "reward", "stop_rate"])
print(
    f"\nbest exact_match {summ['best_exact_match']:.3f}   "
    f"{summ['total_time_s']:.0f}s total"
)

768d x 12L   window 256   batch 8x2   lr 3e-05   flex
        step        comp      prompt exact_match      reward   stop_rate
           0      6.0672      8.7281      0.0000      0.0061      0.0000
          50      0.4321      8.0188      0.4000      0.7365      0.9650
         100      0.0074      8.7969      0.9950      0.9994      1.0000
         150      0.0055      9.1750      0.9900      0.9975      0.9900
         200      0.0018      8.9437      0.9950      0.9994      1.0000
         250      0.0005      9.0594      1.0000      1.0000      1.0000
         300      0.0001      8.8781      1.0000      1.0000      1.0000
         350      0.0004      8.7688      1.0000      1.0000      1.0000
         400      0.0091      8.6031      0.9900      0.9979      1.0000
         450      0.0003      8.7719      1.0000      1.0000      1.0000
         500      0.0045      9.2781      0.9950      0.9994      0.9950
         550      0.0001      9.6687      1.0000      1.0000      1.00

## 2. what masking costs the prompt

`comp` is the loss on completion tokens (trained on). `prompt` is the loss on
prompt tokens, which are masked out of the gradient and only watched. It does not
rise monotonically — it **falls, then climbs past where it started**. **< 1 s.**

Recorded result: from scratch **10.8 → 6.5 → 11.7**; from the pretrained base
**8.73 → 8.02 → 9.82**.

| run            | start | dip       | end   |
| -------------- | ----- | --------- | ----- |
| scratch 11.2M  | 10.82 | 6.48 @100 | 11.70 |
| fineweb 123.6M | 8.73  | 8.02 @50  | 9.82  |

Two legs. Down: training on completions teaches _this data only uses 27 tokens_,
which helps every position including the untrained ones. Up: the model
specialises into a reversal machine and becomes **confidently wrong** about
prompt tokens.

The dip is **0.7 nats from the pretrained base against 4.3 from scratch** — the
base already models text, so there is far less free alphabet-learning to collect.
Neither run reaches the floor: prompt letters are uniform over 26, so ln(26) =
3.26 is the best any model can do there.


In [4]:
for label in ("scratch 11.2M", "fineweb 123.6M"):
    cfg, rows, _ = load_run(label)
    dip = min(rows, key=lambda r: r["prompt"])
    print(
        f"{label}   start {rows[0]['prompt']:.2f} -> dip {dip['prompt']:.2f} "
        f"@{dip['step']} -> end {rows[-1]['prompt']:.2f}"
        f"   (fell {rows[0]['prompt'] - dip['prompt']:.2f} nats first)"
    )
    table(rows, ["step", "comp", "prompt", "exact_match"])
    print()
print(f"floor for a random prompt letter: ln(26) = {math.log(26):.2f}")

scratch 11.2M   start 10.81 -> dip 6.48 @100 -> end 11.70   (fell 4.33 nats first)
        step        comp      prompt exact_match
           0     10.8125     10.8125      0.0000
         100      0.3391      6.4797      0.6500
         200      0.0050      9.7500      1.0000
         300      0.0020     11.0437      1.0000
         400      0.0005     11.7031      1.0000

fineweb 123.6M   start 8.73 -> dip 8.02 @50 -> end 9.82   (fell 0.71 nats first)
        step        comp      prompt exact_match
           0      6.0672      8.7281      0.0000
          50      0.4321      8.0188      0.4000
         100      0.0074      8.7969      0.9950
         150      0.0055      9.1750      0.9900
         200      0.0018      8.9437      0.9950
         250      0.0005      9.0594      1.0000
         300      0.0001      8.8781      1.0000
         350      0.0004      8.7688      1.0000
         400      0.0091      8.6031      0.9900
         450      0.0003      8.7719      1.0000
  

## 3. what the model predicts where it was never trained

Feed a prompt and read the top prediction at each prompt position — positions the
loss never scored. **Loads the checkpoint, ~10 s.**

Recorded result: it collapses onto **an early letter of the word**, at p = 0.8–0.98 — though the first position can be soft.

| prompt   | predictions at prompt positions                   |
| -------- | ------------------------------------------------- |
| `dog>`   | `d`(0.82) `d`(0.97) `d`(0.77)                     |
| `quiz>`  | `q`(0.45) `q`(0.97) `q`(0.92) `u`(0.82)           |
| `zebra>` | `d`(0.22) `z`(0.87) `e`(0.78) `e`(0.93) `e`(0.98) |

Not a clean rule, and not the same failure the small from-scratch model has — that
one echoes _the current token_ at p = 1.00 (`c`→`c`, `a`→`a`, `t`→`t`). The
obvious guess is attention dumping mass on position 0, the sink. **Untested**:
flex and SDPA do not hand back attention weights, so this is a hypothesis, not a
measurement.

This is what the 9.82 prompt loss above looks like from the inside — the
model has not forgotten the alphabet (mass on the 26 letters stays ~1.0), it has
become certain.


In [5]:
task = Reverse()  # the constructor only builds the char table, not the data
tok = task.tok
model, _ = load_checkpoint(CKPT, DEV)
model.eval()
letters = torch.tensor([int(i) for i in task.char_ids], device=DEV)
d = lambda i: tok.decode([int(i)]).replace(ENDOFTEXT, "<eot>")

for word in ["cat", "dog", "quiz", "zebra"]:
    seq = [tok.encode(c)[0] for c in word] + [task.sep_id]
    with torch.no_grad():
        p = model(torch.tensor([seq], device=DEV))[0].float().softmax(-1)
    top = "  ".join(f"{d(p[t].argmax())!r}({p[t].max():.2f})" for t in range(len(word)))
    mass = p[: len(word)][:, letters].sum(-1).min()
    print(f"{word + '>':<8} {top:<46} min letter-mass {mass:.3f}")

cat>     'c'(0.63)  'c'(0.94)  'c'(0.88)                min letter-mass 0.850
dog>     'd'(0.82)  'd'(0.97)  'd'(0.77)                min letter-mass 0.878
quiz>    'q'(0.45)  'q'(0.97)  'q'(0.92)  'u'(0.82)     min letter-mass 0.840
zebra>   'd'(0.22)  'z'(0.87)  'e'(0.78)  'e'(0.93)  'e'(0.98) min letter-mass 0.798


## 4. what a run costs

Three runs on the same task, same 4060. **< 1 s.**

Recorded result: the 123.6M base solves reverse at **lr 3e-5** in ~100 steps;
from scratch needs **lr 1e-3**, and 200 steps.

| run            | base           | lr   | steps to 1.000           | total |
| -------------- | -------------- | ---- | ------------------------ | ----- |
| scratch 11.2M  | none           | 1e-3 | 200                      | 22 s  |
| smoke base 30M | fineweb_smoke  | 3e-5 | not reached (0.655 @400) | 49 s  |
| fineweb 123.6M | fineweb 123.6M | 3e-5 | 250                      | 276 s |

The 30M row is an accident worth keeping: `latest_ckpt("fineweb")` prefix-matched
`fineweb_smoke_*` and finetuned the wrong model. Its base was trained on a
different corpus, so its val loss is not comparable to the 123.6M one — which is
exactly how the bug hid.

**The memory ceiling is the logits, not the weights.** Batch 16 × window 256 OOMs
on 8 GB: `[16, 256, 50259]` in fp32 is ~785 MiB, and the backward wants another.
Halving the batch and accumulating keeps tokens-per-step identical.


In [2]:
print(
    f"{'run':>16}{'window':>8}{'tok/step':>10}{'steps':>7}{'s/step':>9}"
    f"{'total s':>9}{'exact':>8}"
)
for label in RUNS:
    cfg, rows, summ = load_run(label)
    per_step = cfg["batch_size"] * cfg["grad_accum_steps"] * cfg["window"]
    print(
        f"{label:>16}{cfg['window']:>8}{per_step:>10}{cfg['max_steps']:>7}"
        f"{summ['total_time_s'] / cfg['max_steps']:>9.2f}"
        f"{summ['total_time_s']:>9.0f}{summ['best_exact_match']:>8.3f}"
    )

print(
    f"\npeak logits at batch 16, window 256: "
    f"{16 * 256 * 50259 * 4 / 2**20:.0f} MiB in fp32"
)

             run  window  tok/step  steps   s/step  total s   exact
   scratch 11.2M     128      4096    400     0.06       24   1.000
  smoke base 30M     256      4096    400     0.12       49   0.655
  fineweb 123.6M     256      4096   1000     0.28      276   1.000
 instruct 123.6M     512      8192   1000     0.78      779   0.683
 instruct 27M ts     512     16384   2000     0.27      548   0.767
  reverse 27M ts     256      8192   1000     0.11      113   1.000

peak logits at batch 16, window 256: 785 MiB in fp32


## 5. the second rung: TinyStoriesInstruct

Same base, same loop, a different Task -- fields in, a story out. Reverse tested
the trainer; this is the job. **Reads a log, < 1 s.**

Recorded result: **stop_rate 0.00 -> 0.80** is what SFT buys here.

| step | comp  | prompt | words | words_all | stop_rate | story_words |
| ---- | ----- | ------ | ----- | --------- | --------- | ----------- |
| 0    | 2.505 | 4.320  | 0.150 | 0.00      | **0.00**  | 213         |
| 250  | 1.702 | 4.128  | 0.533 | 0.25      | 0.65      | 154         |
| 700  | 1.581 | 4.081  | 0.583 | 0.10      | **1.00**  | 134         |
| 1000 | 1.563 | 4.089  | 0.683 | 0.25      | 0.80      | 154         |

The base never emits eot -- it echoes the Summary and loops `Story:` headers
until the budget runs out. That, not fluency, is what finetuning fixes.

**No U-shape here.** Prompt loss drifts _down_, 4.32 -> 4.09. Reverse's prompt was
half the tokens and a random string, so specialising on the completion made the
prompt unpredictable. Here the prompt is 18% of tokens and it is English that
overlaps the story's own vocabulary, so the two halves are not fighting.

Stories shorten 213 -> 154 words: the base rambles to the budget, the finetune
learns TinyStories length. `words_all` at n=20 is worth about ±0.1 -- the run's
0.25 and a 40-sample re-measure at 0.10 are the same number.


In [ ]:
cfg, rows, summ = load_run("instruct 123.6M")
print(
    f"window {cfg['window']}   batch {cfg['batch_size']}x{cfg['grad_accum_steps']}"
    f"   lr {cfg['lr']}   {summ['total_time_s'] / 60:.1f} min"
)
table(
    rows, ["step", "comp", "prompt", "words", "words_all", "stop_rate", "story_words"]
)

window 512   batch 4x4   lr 3e-05   13.0 min
        step        comp      prompt       words   words_all   stop_rate story_words
           0      2.5047      4.3203      0.1500      0.0000      0.0000    212.7000
          50      1.9551      4.2547      0.3667      0.0500      0.1500    185.8500
         100      1.8375      4.1867      0.4333      0.1500      0.2000    178.2000
         150      1.7762      4.1328      0.4167      0.0500      0.4500    165.5000
         200      1.7297      4.1500      0.4833      0.1000      0.4500    165.2000
         250      1.7016      4.1281      0.5333      0.2500      0.6500    154.2000
         300      1.6770      4.1375      0.5000      0.1500      0.5500    172.0500
         350      1.6562      4.1156      0.5833      0.2000      0.6000    167.2500
         400      1.6395      4.1180      0.6000      0.1500      0.6000    163.5500
         450      1.6242      4.0938      0.6000      0.2500      0.7500    158.1000
         500      1.

## 6. greedy decoding understates the scoreboard

`evaluate` decodes greedily so the number is reproducible. Greedy is also the
worst case for repetition -- this base loops 100% of the time at temperature 0
and 4% at 1.0 -- so the scoreboard is measuring the model _and_ the decoder.
40 prompts per setting. **~2 min.**

Recorded result: **stop_rate 0.725 greedy vs 0.925 at t=0.8/p=0.95.**

| decoding      | words | words_all | stop_rate | story_words |
| ------------- | ----- | --------- | --------- | ----------- |
| greedy        | 0.550 | 0.100     | 0.725     | 152         |
| t=0.8, p=0.95 | 0.633 | 0.200     | **0.925** | 155         |
| t=1.0         | 0.567 | 0.175     | 0.775     | 167         |

Half the failures the greedy score counts are the decoder's, not the weights'.
Worth knowing before reading a number like `words_all 0.10` as a training
shortfall -- and before using it as the DPO/GRPO target, where the sampling
distribution is the thing being optimised.


In [ ]:
task = Instruct()
model, _ = load_checkpoint(CKPT_INSTRUCT, DEV)

print(
    f"{'decoding':<16}{'words':>8}{'words_all':>11}{'stop_rate':>11}{'story_words':>13}"
)
for label, kw in [
    ("greedy", {}),
    ("t=0.8, p=0.95", {"temperature": 0.8, "top_p": 0.95}),
    ("t=1.0", {"temperature": 1.0}),
]:
    s = task.evaluate(model, n=40, **kw)
    print(
        f"{label:<16}{s['words']:>8.3f}{s['words_all']:>11.3f}"
        f"{s['stop_rate']:>11.3f}{s['story_words']:>13.1f}"
    )

decoding           words  words_all  stop_rate  story_words


greedy             0.550      0.100      0.725        152.2


t=0.8, p=0.95      0.633      0.200      0.925        154.7


t=1.0              0.567      0.175      0.775        166.8


## 7. what the models actually write

The numbers above, as output. The same prompts through the base and through each
finetune, greedy, at the budget the scoreboard uses. **Loads three checkpoints,
~1 min.**

Recorded result -- reverse, l+1 tokens:

| prompt    | base                     | after SFT       |
| --------- | ------------------------ | --------------- |
| `cat>`    | `'\n- The first'`        | `'tac<eot>'`    |
| `zebra>`  | `'\n- "The Great War'`   | `'arbez<eot>'`  |
| `puzzle>` | `'\n- "The Jews of the'` | `'elzzup<eot>'` |

The base does not fail at reversing -- it does not see a task at all. `cat>` is
web text, so it writes a bullet and a headline. The finetune reverses and stops
on the token after, which is the whole of `exact_match 1.000`.

Recorded result -- instruct, a prompt asking for a story using _escape, war,
tall_. Three continuations, same prompt:

|                         | stops?                                                                                                                               |
| ----------------------- | ------------------------------------------------------------------------------------------------------------------------------------ |
| base, greedy            | no -- echoes the Summary, turns Ben into "a tall tree", then loops `Story:` headers to the budget                                    |
| after SFT, greedy       | no -- invents characters, opens with _One day_, uses two of the three words, then degenerates into _"it is very tall and very tall"_ |
| after SFT, t=0.8 p=0.95 | **yes** -- twins in a forest, dialogue, and an eot                                                                                   |

The last two rows are **the same weights**. Greedy walks into the repetition
loop; sampling does not. That is section 6's `0.725` vs `0.925` stop_rate seen
from the inside, and it is the sharpest argument for what comes next: the model
already assigns probability to a story that terminates, and nothing in SFT
prefers it over the one that does not. Moving mass between two samples the model
already writes is exactly DPO's move.


In [ ]:
base, _ = load_checkpoint(
    latest_ckpt("fineweb", max_val_loss=6.0), DEV, attention="flex"
)

rev = Reverse()
rev_tuned, _ = load_checkpoint(CKPT, DEV)
print("=== reverse ===")
for word in ["cat", "zebra", "puzzle"]:
    # char by char: tok.encode("cat>") would merge, and reversal is position work
    ids = [int(rev.char_ids[ord(c) - ord("a")]) for c in word] + [rev.sep_id]
    x = torch.tensor([ids], device=DEV)
    for label, m in (("base", base), ("sft", rev_tuned)):
        out = generate(m, x, len(word) + 1, temperature=0.0)
        got = rev.tok.decode(out[0, x.size(1) :].tolist()).replace(ENDOFTEXT, "<eot>")
        print(f"  {word + '>':<9} {label:<5} {got!r}")

ins = Instruct()
ins_tuned, _ = load_checkpoint(CKPT_INSTRUCT, DEV)
prompt = ins.prompts()[0]
x = torch.tensor([ins.tok.encode(prompt)], device=DEV)
print(f"\n=== instruct ===\n{prompt}")
for label, m, kw in (
    ("--- base, greedy ---", base, {}),
    ("--- after SFT, greedy ---", ins_tuned, {}),
    ("--- after SFT, t=0.8 p=0.95 ---", ins_tuned, {"temperature": 0.8, "top_p": 0.95}),
):
    kw = {"temperature": 0.0} | kw
    if kw["temperature"]:  # seeded, so the archive shows the same story twice
        kw["generator"] = torch.Generator(device=DEV).manual_seed(0)
    out = generate(m, x, 200, use_cache=True, **kw)
    text = ins.tok.decode(out[0, x.size(1) :].tolist())
    print(f"\n{label}\n{text.split(ENDOFTEXT)[0][:520]}")
    print(f"[stopped: {ENDOFTEXT in text}]")

=== reverse ===
  cat>      base  '\n- The first'
  cat>      sft   'tac<eot>'
  zebra>    base  '\n- "The Great War'
  zebra>    sft   'arbez<eot>'
  puzzle>   base  '\n- "The Jews of the'
  puzzle>   sft   'elzzup<eot>'



=== instruct ===
Summary: Tom and Lily escape from a scary house and find a cozy tree house in the forest where they meet Ben, who invites them to stay with him.
Features: Dialogue
Words: escape, war, tall
Story:




--- base, greedy ---
Story: Tom and Lily escape from a scary house and find a cozy tree house in the forest where they meet Ben, who is a tall tree. Ben and Lily escape from a scary house and find a cozy tree house in the forest where they meet Ben and Lily.
Story: Tom and Lily escape from a scary house and find a cozy tree house in the forest where they meet Ben and Lily.
Story: Tom and Lily escape from a scary house and find a cozy tree house in the forest where they meet Ben and Lily.
Story: Tom and Lily escape from a scary house an
[stopped: False]



--- after SFT, greedy ---
Tom and Lily were friends who liked to play in the forest. They liked to run and jump and jump and jump. They also liked to play in the trees and the trees.
One day, they saw a big house with a big tree. They wanted to escape from the house. They ran to the house and tried to climb it. But the house was too tall and too tall.
"Look, Lily, a big house!" Tom said. "It is very tall and tall. It is very tall and very tall. It is very tall and very tall. It is very tall and very tall. It is very tall and very tall. It i
[stopped: False]



--- after SFT, t=0.8 p=0.95 ---
Tom and Lily were twins who liked to play in the forest. They were always fast and did not want to run or run. One day, they saw a big tree with tall branches. They wanted to escape, but they did not want to.
"Yuck!" Tom said, "I want to escape, but not to get out."
Lily said, "No, you are too small. You are too small. You can't escape. You must find a way to stay with me."
Lily and Tom were excited. They ran to the tree house and found a big tree. They pulled and pulled and pulled and pulled. They were afraid.
"Do
[stopped: True]


## 8. the same rung from a TinyStories base

Section 5 finetuned the 123.6M FineWeb base on instruct. This is the same task
and the same loop from the 27M TinyStories base (val 1.3986, bpc 0.4954), which
is what the rung was always meant to run on. lr 1e-4, chosen by 200-step probes
against 3e-5 and 3e-4. **Reads a log, < 1 s.**

Recorded result: **9.1 min**, and what moves is the word constraints.

|             | base   | after SFT |
| ----------- | ------ | --------- |
| comp        | 1.5043 | 1.0945    |
| prompt      | 4.0922 | 4.6937    |
| words       | 0.125  | 0.742     |
| words_all   | 0.000  | **0.425** |
| stop_rate   | 0.575  | 0.925     |
| story_words | 56.5   | 153.8     |

**This base fails the opposite way to FineWeb's.** That one rambled to the token
budget and never stopped. This one stops far too early — 56 words against the
corpus's real mean of 159. It knows how to end a story; it does not know it has
been asked for a whole one.

Most of the gain arrives by step 200 (`comp 1.1477` against `1.0945` at 2000),
the same shape the old track saw: 1600 steps reached 1.0355 and 12x more compute
reached 1.0003.

Prompt loss **rises** here, 4.09 -> 4.69, where on the FineWeb base it fell. A
plausible reason is that this base has never seen field scaffolding like
`Words:`/`Summary:`, so specialising on stories costs it there, while FineWeb had
seen plenty. Untested.


In [ ]:
cfg, rows, summ = load_run("instruct 27M ts")
print(
    f"{cfg['n_embed']}d x {cfg['n_layer']}L   window {cfg['window']}   "
    f"batch {cfg['batch_size']}x{cfg['grad_accum_steps']}   lr {cfg['lr']}   "
    f"{summ['total_time_s'] / 60:.1f} min"
)
table(
    rows, ["step", "comp", "prompt", "words", "words_all", "stop_rate", "story_words"]
)

512d x 8L   window 512   batch 32x1   lr 0.0001   9.1 min
        step        comp      prompt       words   words_all   stop_rate story_words
           0      1.5043      4.0922      0.1250      0.0000      0.5750     56.4750
         200      1.1477      4.6594      0.7000      0.2250      0.8750    153.3000
         400      1.1342      4.7391      0.7083      0.3250      0.8750    151.0000
         600      1.1240      4.6828      0.7667      0.4000      0.9500    150.2500
         800      1.1156      4.6516      0.7333      0.3500      0.8750    155.9250
        1000      1.1105      4.7047      0.7333      0.4000      0.8750    149.7000
        1200      1.1043      4.7031      0.7250      0.3250      0.8750    149.9000
        1400      1.1014      4.6969      0.7500      0.4000      0.8750    151.9750
        1600      1.0973      4.6906      0.7417      0.3750      0.9000    153.5750
        1800      1.0945      4.6891      0.7333      0.4000      0.8750    149.8250
       

## 9. one prompt, four samples, the same weights

The scoreboard is a mean, and a mean hides the spread that DPO exists to
exploit. One held-out prompt: greedy, then three seeds at t=0.8 p=0.95.
**Loads a checkpoint, ~30 s.**

Recorded result: **reward 0.33 to 1.00, and stopping is close to a coin flip.**

| sample   | stopped | reward   |
| -------- | ------- | -------- |
| greedy   | yes     | 0.67     |
| t=0.8 #0 | no      | 0.33     |
| t=0.8 #1 | **yes** | **1.00** |
| t=0.8 #2 | no      | 0.67     |

Read the stories and a second failure appears that no metric here catches. The
prompt asks for _meet, waffle, new_ **and** a summary — Lily is bitten by a dog
and goes to hospital for stitches. Greedy writes a clean, terminated story about
eating a waffle and playing with a toy car: no dog, no hospital. **It satisfies
the Words and ignores the Summary**, because the Words are what the reward
measures. Anything scored by string match gets answered by string match.

Sample #1 does everything — all three words, the dog, the hospital, and an eot.
It came from the same weights as #0, which follows the plot but runs past the
budget without stopping. That gap is the entire setup for DPO: the good
completion is already in the distribution, and nothing in SFT prefers it.

Note the checkpoint these come from is **step 600, not 2000**: `sft.py` selects
on the task's first metric, `words`, which peaked at 0.767 there by noise
(n=40 carries about +/-0.1). So the kept model has `comp 1.124` against the final
step's 1.0945 — a noisy generative metric picking a worse model on loss. Select
on `comp`, or raise `eval_n`.


In [ ]:
task = Instruct()
model, meta = load_checkpoint(CKPT_TS_INSTRUCT, DEV)
prompt = task.prompts()[1]
x = torch.tensor([task.tok.encode(prompt)], device=DEV)
print(prompt)

runs = [("greedy", {"temperature": 0.0})]
runs += [
    (f"t=0.8 p=0.95 #{i}", {"temperature": 0.8, "top_p": 0.95, "seed": i})
    for i in range(3)
]
for label, kw in runs:
    kw = dict(kw)
    seed = kw.pop("seed", 0)
    if kw["temperature"]:
        kw["generator"] = torch.Generator(device=DEV).manual_seed(seed)
    out = generate(model, x, 300, use_cache=True, **kw)
    text = task.tok.decode(out[0, x.size(1) :].tolist())
    print(
        f"\n--- {label} | stopped {ENDOFTEXT in text} "
        f"| reward {task.reward(prompt, text):.2f} ---"
    )
    print(text.split(ENDOFTEXT)[0])

Words: meet, waffle, new
Summary: Lily wants a waffle but gets bitten by a dog while playing with a toy car and has to go to the hospital for stitches.
Story:




--- greedy | stopped True | reward 0.67 ---
Once upon a time, there was a little girl named Lily. She loved waffles more than anything in the world. One day, her mom made her a new waffle for breakfast. It was so yummy!
Lily's mom asked her if she wanted to try it. Lily said yes and her mom gave her a waffle. Lily took a bite and made a funny face. "This waffle is so yummy!" she said.
Later that day, Lily went to the park with her friends. They played on the swings and the slide. Lily's friend, Timmy, wanted to play with her toy car. "Can I play with your car?" Timmy asked. "Sure!" Lily said. They played together and had so much fun. When it was time to go home, Lily said, "I had a great time with you today. Let's play again soon!"



--- t=0.8 p=0.95 #0 | stopped False | reward 0.33 ---
Lily likes to play with her toys. She has many toys, but her favorite is a big waffle. She likes to put waffles on the waffle and eat them every day. She thinks they are yummy and warm.
One day, Lily is playing with her waffle in the living room. She wants to see if it is good for her mouth. She takes a big bite and says, "Mmm, this is good!" She smiles and shares the waffle with her toy car.
But then, a big dog comes into the living room. The dog is old and has a long tail. He sees the waffle and thinks it is a toy. He runs towards it and bites it. The waffle falls on the floor and makes a loud noise. Lily is scared and cries, "No, no, no!"
Mom hears the noise and comes into the living room. She sees the dog and shakes her head. She says, "Oh no, you are a bad dog. You should not play with the waffle. You can't eat it. It is not a toy. It is a waffle."
She picks up the waffle and puts it away. She hugs Lily and says, "I'm sorry, s


--- t=0.8 p=0.95 #1 | stopped True | reward 1.00 ---
Once upon a time, there was a little girl named Lily. She loved waffles and ate them every day. One day, her mom made her a new waffle that was very yummy. Lily ate it all up and wanted more. Her mom said no, but Lily didn't listen and ate another waffle.
Later that day, Lily went to the park to play. She saw a big dog and wanted to meet him. But the dog was too big and scary. Lily was scared and started to cry. Suddenly, she heard a loud noise and felt a sharp pain in her toe. She had fallen and hurt herself.
Lily's mom rushed her to the hospital and the doctor said she needed a bandage. Lily was sad and wished she had listened to her mom. She learned that sometimes, things that seem different can be very bad.



--- t=0.8 p=0.95 #2 | stopped False | reward 0.67 ---
Lily and Ben were friends who liked to play together. One day, they went to the park with their moms. There, they saw a big dog with a brown coat and a long tail. The dog was eating a waffle.
"Can we pet the dog?" Lily asked Ben.
"OK, but be careful. He is very big and strong," Ben said.
They went to the dog and smiled. The dog licked their faces and wagged his tail. Lily and Ben laughed and petted the dog.
But then, they heard a loud bark. A big dog was running towards them. The dog was not friendly. It was angry and scary.
"Help! Help!" Lily and Ben screamed.
The dog stopped and looked at them. It did not want to hurt them. It just wanted to have fun.
The dog licked their faces again. Lily and Ben felt silly. They thought it was funny.
"Sorry, dog. We did not mean to scare you. Are you OK?" Lily said.
The dog barked again and wagged its tail again. It was not so scary after all.
Lily and Ben laughed. They liked the dog. They pett

## 10. the other task from the same base

Section 1 solved `Reverse` from the 123.6M FineWeb base. This is the same task
and the same loop from the 27M TinyStories base, so the LoRA rung has a full-SFT
number to be measured against on _both_ tasks, not just instruct. lr 1e-4, the
value section 8 measured for this base. **Reads a log, < 1 s.**

Recorded result: **1.000 exact match by step 300, 113 s** -- and a base that is
_worse than uniform_ on the prompt before training starts.

| step | comp   | prompt  | exact_match | stop_rate |
| ---- | ------ | ------- | ----------- | --------- |
| 0    | 8.1094 | 9.6969  | 0.000       | 0.005     |
| 100  | 0.2423 | 9.5437  | 0.420       | 0.450     |
| 200  | 0.0096 | 9.5719  | 0.990       | 1.000     |
| 300  | 0.0024 | 9.6969  | **1.000**   | 1.000     |
| 1000 | 0.0000 | 10.2937 | 1.000       | 1.000     |

**ln(V) is the line to read a starting loss against**, and for this vocab that is
ln(4097) = 8.32, not FineWeb's ln(50259) = 10.82. Same task, same metric, two
numbers that are two nats apart for no reason but the tokenizer.

| base           | vocab | ln(V) | comp @0   | prompt @0 |
| -------------- | ----- | ----- | --------- | --------- |
| scratch 11.2M  | 50259 | 10.82 | -0.01     | -0.01     |
| fineweb 123.6M | 50259 | 10.82 | **-4.76** | -2.10     |
| reverse 27M ts | 4097  | 8.32  | -0.21     | **+1.38** |

Two things fall out of that last row.

**Pretraining on TinyStories buys almost nothing for reversal** -- 0.21 nats under
uniform, against FineWeb's 4.76. FineWeb-Edu has seen spelling, acrostics and
letter-by-letter text; TinyStories is 4-year-old vocabulary in whole words and has
seen none of it.

**And it starts _above_ uniform on the prompt**, 1.38 nats worse than guessing. A
story-only model is confidently certain that `qxzv` cannot happen, where the
FineWeb base was 2.10 nats _below_ uniform on the same tokens. This is section 2's
point in its sharpest form: the prompt half is where a specialised base gets
punished, and here it is punished before the finetune has taken a step.

It still solves the task in 300 steps against FineWeb's 250, on a model 4.5x
smaller and in 113 s against 276. Different lr and different size, so this is not
a controlled comparison -- but whatever the base contributes here, it is plainly
not prior knowledge of the task.

**Section 2's dip barely happens, and what follows it is noise rather than drift.**
Prompt loss falls 0.15 nats by step 100 -- against 0.71 for FineWeb and 4.33 from
scratch -- and `eval_interval` is 100 here where it was 50 there, so a sharper,
earlier dip would simply have been stepped over. Past step 300 the number swings
over **1.11 nats** (9.22 at step 700, 10.33 at step 600) while `comp` sits at
0.0000. Those evals run on _identical_ val windows, so that is the model's
prompt-side predictions thrashing, not sampling noise. Read the end value as
"somewhere around 10.2", not as a trend.


In [3]:
cfg, rows, summ = load_run("reverse 27M ts")
print(
    f"{cfg['n_embed']}d x {cfg['n_layer']}L   window {cfg['window']}   "
    f"batch {cfg['batch_size']}x{cfg['grad_accum_steps']}   lr {cfg['lr']}   "
    f"{summ['total_time_s']:.0f}s total"
)
table(rows, ["step", "comp", "prompt", "exact_match", "reward", "stop_rate"])

# a base only helps where it starts below ln(V) -- uniform over ITS OWN vocab,
# which is what makes 8.11 and 6.07 comparable despite being 2 nats apart
print(f"\n{'run':>16}{'vocab':>8}{'ln(V)':>8}{'comp@0':>9}{'prompt@0':>10}")
for label in ("scratch 11.2M", "fineweb 123.6M", "reverse 27M ts"):
    c, r, _ = load_run(label)
    lnv = math.log(c["vocab_size"])
    gap = f"{r[0]['comp'] - lnv:+.2f} / {r[0]['prompt'] - lnv:+.2f}"
    print(
        f"{label:>16}{c['vocab_size']:>8}{lnv:>8.2f}{r[0]['comp']:>9.2f}"
        f"{r[0]['prompt']:>10.2f}{gap:>16}  vs ln(V)"
    )

# split the two legs at the step the task is solved: before it, the section 2
# story; after it, whatever the prompt side does once comp has nothing left
solved = next(r["step"] for r in rows if r["exact_match"] == 1.0)
before = [r for r in rows if r["step"] <= solved]
after = [r for r in rows if r["step"] > solved]
dip = min(before, key=lambda r: r["prompt"])
lo, hi = min(after, key=lambda r: r["prompt"]), max(after, key=lambda r: r["prompt"])
print(f"\nexact_match 1.000 first reached at step {solved}")
print(
    f"  to step {solved}: prompt {rows[0]['prompt']:.2f} -> dip {dip['prompt']:.2f}"
    f" @{dip['step']}   (fell {rows[0]['prompt'] - dip['prompt']:.2f} nats)"
)
print(
    f"  after:       {lo['prompt']:.2f} @{lo['step']} to {hi['prompt']:.2f}"
    f" @{hi['step']}, ends {rows[-1]['prompt']:.2f}"
    f"   ({hi['prompt'] - lo['prompt']:.2f} nats of scatter on identical windows)"
)

512d x 8L   window 256   batch 32x1   lr 0.0001   113s total
        step        comp      prompt exact_match      reward   stop_rate
           0      8.1094      9.6969      0.0000      0.0071      0.0050
         100      0.2423      9.5437      0.4200      0.8869      0.4500
         200      0.0096      9.5719      0.9900      0.9975      1.0000
         300      0.0024      9.6969      1.0000      1.0000      1.0000
         400      0.0023      9.9344      1.0000      1.0000      1.0000
         500      0.0004     10.2531      1.0000      1.0000      1.0000
         600      0.0003     10.3281      1.0000      1.0000      1.0000
         700      0.0001      9.2156      1.0000      1.0000      1.0000
         800      0.0000     10.1781      1.0000      1.0000      1.0000
         900      0.0000     10.0875      1.0000      1.0000      1.0000
        1000      0.0000     10.2937      1.0000      1.0000      1.0000

             run   vocab   ln(V)   comp@0  prompt@0
   scratch